# Фильтрация дампа Open Food Facts по России

Этот ноутбук **вручную** запускает фильтрацию полного дампа OFF по России, чтобы вы видели прогресс в реальном времени.

Дамп `data/downloads/openfoodfacts-products.jsonl` (~74 ГБ) уже должен быть скачан скриптом `scripts/02_download_off.py`.

**Что делает:** проходит по всем ~4.6 млн строк дампа, оставляет продукты с тегом `en:russia` и сохраняет в `data/processed/off_ru_foods.csv`.
Сохраняются ВСЕ РФ-продукты — даже если у них нет нутриентов (важно получить полный список).

Время работы: ~20-30 минут. Прогресс виден в полоске tqdm под ячейкой.

## 1. Проверка исходного дампа

Убедимся, что дамп на месте и видно его размер.

In [ ]:
import os, sys, json, csv, time
sys.path.insert(0, os.path.abspath('..'))

DUMP = os.path.abspath('../data/downloads/openfoodfacts-products.jsonl')
OUT  = os.path.abspath('../data/processed/off_ru_foods.csv')

if os.path.exists(DUMP):
    size_gb = os.path.getsize(DUMP) / 1073741824
    print(f'✅ Дамп найден: {DUMP}')
    print(f'   размер: {size_gb:.1f} ГБ')
else:
    print(f'❌ Дамп НЕ найден: {DUMP}')
    print('   Сначала запустите scripts/02_download_off.py')
print(f'\nРезультат будет сохранён в: {OUT}')

## 2. Быстрая проверка структуры (1 строка дампа)

Посмотрим, какие поля есть в записях — чтобы убедиться, что нутриенты читаются из вложенного `nutriments`.

In [ ]:
with open(DUMP, 'r', encoding='utf-8', errors='ignore') as f:
    first = json.loads(f.readline())
print('Ключи верхнего уровня (первые 20):', sorted(first.keys())[:20], '...')
print()
nm = first.get('nutriments') or {}
print(f"nutriments: тип={type(nm).__name__}, ключей={len(nm)}")
if isinstance(nm, dict) and nm:
    print('пример нутриентов:', dict(list(nm.items())[:8]))
print()
print(f"продукт: {first.get('product_name','?')[:50]}")
print(f"страна: {first.get('countries','?')[:50]}")

## 3. Загрузить логику из скрипта

Берём функцию `row_from_product` и константы прямо из `scripts/03_filter_off_ru.py` — единый источник правды, без дублирования.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    'filter_off_ru', os.path.abspath('../scripts/03_filter_off_ru.py'))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

RU_TAG = mod.RU_TAG
COLUMNS = mod.COLUMNS
row_from_product = mod.row_from_product
print('✅ Логика загружена из скрипта.')
print('Фильтрую по тегу:', RU_TAG)
print('Колонки:', COLUMNS)

## 4. Запуск фильтрации

Запустите ячейку 4a, потом 4b (`Shift+Enter`). Прогресс виден в полоске tqdm под 4b. ~4.6 млн строк, займёт ~20-30 мин.

In [ ]:
# 4a. Оцениваем количество строк (один быстрый проход по байтам, ~3-4 мин)
print('Считаю строки в дампе ...')
t0 = time.time()
with open(DUMP, 'rb') as f:
    total = sum(1 for _ in f)
print(f'Строк: {total:,}  (за {time.time()-t0:.0f}s)')

In [ ]:
# 4b. Основная фильтрация — с живым прогрессом
from tqdm import tqdm

KBJU_COLS = ('kcal', 'protein_g', 'fat_g', 'carbs_g')
NUTRI_COLS = KBJU_COLS + ('fiber_g','sugars_g','sat_fat_g','sodium_mg',
                          'potassium_mg','calcium_mg','iron_mg','vitc_mg')

n_total = n_ru_all = n_written = n_no_id = 0
n_with_any_nutri = n_with_full_kbju = 0
t1 = time.time()

os.makedirs(os.path.dirname(OUT), exist_ok=True)
with open(DUMP, 'r', encoding='utf-8', errors='ignore') as src, \
     open(OUT, 'w', encoding='utf-8-sig', newline='') as dst, \
     tqdm(total=total, unit='строк', ncols=90, desc='filter') as bar:
    writer = csv.DictWriter(dst, fieldnames=COLUMNS)
    writer.writeheader()
    for line in src:
        n_total += 1
        bar.update(1)
        try:
            p = json.loads(line)
        except json.JSONDecodeError:
            continue
        countries = p.get('countries_tags') or []
        if RU_TAG not in countries:
            continue
        n_ru_all += 1
        row = row_from_product(p)
        if row is None:
            n_no_id += 1
            continue
        if any(row.get(c) is not None for c in NUTRI_COLS):
            n_with_any_nutri += 1
        if all(row.get(c) is not None for c in KBJU_COLS):
            n_with_full_kbju += 1
        writer.writerow(row)
        n_written += 1

elapsed = time.time() - t1
print(f'\nГотово за {elapsed/60:.1f} мин')

## 5. Статус — что получилось

In [ ]:
print('=' * 55)
print('ИТОГИ ФИЛЬТРАЦИИ')
print('=' * 55)
print(f'Всего строк в дампе:           {n_total:>9,}')
print(f'Записей с тегом РФ (все):      {n_ru_all:>9,}')
print(f'  отброшено (без кода+назв.):  {n_no_id:>9,}')
print(f'Записано в CSV:                {n_written:>9,}')
if n_written:
    print(f'  с любым нутриентом:          {n_with_any_nutri:>9,}  '
          f'({n_with_any_nutri/n_written*100:.0f}%)')
    print(f'  с полным КБЖУ:               {n_with_full_kbju:>9,}  '
          f'({n_with_full_kbju/n_written*100:.0f}%)')
print()
print(f'Файл: {OUT}')
print(f'Размер: {os.path.getsize(OUT)/1048576:.1f} МБ')

## 6. Заглянуть в результат

In [ ]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 35)

df = pd.read_csv(OUT)
print(f'Строк: {len(df)}, колонок: {df.shape[1]}')
print()
print('Заполненность по колонкам:')
print((df.notna().mean() * 100).round(0).astype(int).astype(str) + '%')
print()
df.head(10)

In [ ]:
# Демо-поиск российских брендов
for q in ['бородинский', 'простоквашино', 'савушкин', 'слобода']:
    found = df[df['name'].str.contains(q, case=False, na=False)]
    print(f'«{q}»: {len(found)} продуктов')